# Lab 04 - Transformación de Datos (Medallion Architecture)

**Objetivos:**
- Implementar arquitectura Bronze → Silver → Gold
- Data quality checks
- Delta Lake operations (MERGE, OPTIMIZE, Z-ORDER)
- Time travel

## Parte 1: Capa Bronze - Ingesta Raw

In [ ]:
# =============================================================================
# FUNCIÓN GENERADORA DE DATOS SINTÉTICOS - VENTAS
# =============================================================================
# Objetivo: Simular sistema de ventas para implementar Medallion Architecture
# Bronze → Silver → Gold

# Importaciones necesarias
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

def generate_sales_data(num_records=1000, batch_id="BATCH_001"):
    """
    Genera datos sintéticos de transacciones de ventas.
    
    Simula un sistema real donde:
    - Pueden llegar datos duplicados (errores de red, reintentos)
    - Datos vienen "crudos" (sin limpiar ni estandarizar)
    - Se necesita metadata de procesamiento (auditoría)
    
    Args:
        num_records (int): Número de transacciones a generar
        batch_id (str): Identificador del lote de carga
    
    Returns:
        DataFrame: Datos RAW con metadata de ingesta
    """
    # Generar fechas de los últimos 30 días
    dates = [(datetime.now() - timedelta(days=x)) for x in range(30)]
    
    # Catálogos de productos y regiones
    products = ["LAPTOP", "MOUSE", "KEYBOARD", "MONITOR", "WEBCAM"]
    regions = ["NORTH", "SOUTH", "EAST", "WEST"]
    
    # Generar transacciones aleatorias

    data = []# 🥇 GOLD: Agregaciones de negocio, listas para analytics/BI

    for i in range(num_records):# 🥈 SILVER: Datos limpios, deduplicados, validados

        data.append({# 🥉 BRONZE: Datos RAW (tal como llegan, con metadata)

            # ID único de transacción (formato TXN000001)# 🏛️ ARQUITECTURA MEDALLION:

            "transaction_id": f"TXN{i:06d}",

            print("✅ Función de generación de datos creada")

            # Fecha aleatoria dentro de los últimos 30 días

            "date": dates[random.randint(0, len(dates)-1)].strftime("%Y-%m-%d"),    return df_bronze

                

            # Código de producto (P100-P999)    # - Linaje: Rastrear origen y transformaciones de los datos

            "product_code": f"P{random.randint(100, 999)}",    # - Deduplicación: Quedarse con el registro más reciente

                # - Auditoría: ¿De dónde vino este dato?

            # Nombre de producto del catálogo    # - Troubleshooting: ¿Cuándo llegó este dato erróneo?

            "product_name": random.choice(products),    # 🎖️ BEST PRACTICE: SIEMPRE agregar metadata en Bronze para:

                

            # Región de venta        )

            "region": random.choice(regions),            lit(batch_id)         # Permite rastrear cuándo llegó cada dato

                        "_batch_id",          # Identificador del lote de carga

            # Monto de la venta ($50 - $1000)        .withColumn(

            "amount": round(random.uniform(50, 1000), 2),        ) \

                        lit("sales_system")   # lit() crea columna con valor constante

            # Cantidad de unidades (1-10)            "_source",            # Sistema de origen del dato

            "quantity": random.randint(1, 10),        .withColumn(

                    ) \

            # ID del cliente (CUST1000-CUST9999)            current_timestamp()   # Fecha/hora actual del sistema

            "customer_id": f"CUST{random.randint(1000, 9999)}",            "_ingested_at",      # Timestamp de cuándo se ingirió el dato

                    .withColumn(

            # Estado de la transacción    df_bronze = df \

            "status": random.choice(["completed", "pending", "cancelled"])    # Estas columnas son CLAVE para auditoría y troubleshooting:

        })    # =========================================================================

        # AGREGAR METADATA DE INGESTA (PATRÓN BRONZE)

    # Crear DataFrame desde lista de diccionarios    # =========================================================================

    df = spark.createDataFrame(data)    

In [ ]:
# Generar primer lote
df_batch1 = generate_sales_data(500, "BATCH_001")
print(f"✅ Batch 1: {df_batch1.count()} registros")
display(df_batch1.limit(10))

In [ ]:
# Guardar en Bronze
table_bronze = "lab04_bronze_sales"

df_batch1.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(table_bronze)

print(f"✅ Datos guardados en tabla Bronze: {table_bronze}")

In [ ]:
# Generar segundo lote con duplicados
df_batch2 = generate_sales_data(300, "BATCH_002")

# Agregar algunos duplicados del batch 1
duplicates = df_batch1.limit(20).select(
    "transaction_id", "date", "product_code", "product_name", 
    "region", "amount", "quantity", "customer_id", "status"
).withColumn("_ingested_at", current_timestamp()) \
 .withColumn("_source", lit("sales_system")) \
 .withColumn("_batch_id", lit("BATCH_002"))

df_batch2_with_dups = df_batch2.union(duplicates)

df_batch2_with_dups.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(table_bronze)

print(f"✅ Batch 2 agregado (con duplicados)")

In [ ]:
# Verificar Bronze
df_bronze = spark.table(table_bronze)
total = df_bronze.count()
duplicates_count = df_bronze.groupBy("transaction_id").count().filter("count > 1").count()

print(f"📊 Total registros en Bronze: {total}")
print(f"⚠️  Transacciones duplicadas: {duplicates_count}")

## Parte 2: Capa Silver - Limpieza y Deduplicación

In [ ]:
# =============================================================================
# DEDUPLICACIÓN USANDO WINDOW FUNCTIONS
# =============================================================================
# Objetivo: Eliminar registros duplicados, quedando con el más reciente
# Problema: En sistemas reales pueden llegar datos duplicados por:
#   - Reintentos de red
#   - Errores en sistemas fuente
#   - Múltiples fuentes reportando el mismo evento

# Importar Window class para operaciones de ventana
from pyspark.sql.window import Window

# 3. Por columna específica: orderBy con criterio de negocio (ej: status priority)

# =========================================================================# 2. Más antiguo: orderBy(col("_ingested_at").asc()) para el primero

# PASO 1: Definir la "ventana" (Window)# 1. Más reciente (esta): Cuando queremos el último estado

# =========================================================================# ⚖️ ¿QUÉ ESTRATEGIA DE DEDUPLICACIÓN USAR?

# Una Window agrupa registros y permite aplicar funciones sobre esos grupos

print(f"✅ Deduplicación: {df_bronze.count()} → {df_dedup.count()} registros")

# partitionBy: Agrupar por transaction_id (cada ventana = 1 transaction_id)

# orderBy: Dentro de cada ventana, ordenar por _ingested_at descendente# Para deduplicación, row_number() es lo correcto

#          (más reciente primero)# - dense_rank(): Puede tener empates, NO salta números (1, 1, 2, 3...)

window_spec = Window.partitionBy("transaction_id").orderBy(col("_ingested_at").desc())# - rank(): Puede tener empates, salta números (1, 1, 3, 4...)

# - row_number(): Siempre 1, 2, 3... (sin empates)

# 📊 EXPLICACIÓN VISUAL:# 💡 ALTERNATIVA: rank() vs row_number() vs dense_rank()

# Si tenemos transaction_id "TXN000001" ingresada 3 veces:

#   TXN000001 | 2026-05-22 10:00:00  ← row_num = 1 (✅ más reciente, MANTENER)    .drop("row_num")                # Eliminar columna auxiliar (ya no la necesitamos)

#   TXN000001 | 2026-05-21 15:30:00  ← row_num = 2 (❌ duplicado antiguo)    ) \

#   TXN000001 | 2026-05-20 09:15:00  ← row_num = 3 (❌ duplicado antiguo)        col("row_num") == 1         # Quedarnos SOLO con la fila #1 de cada ventana

    .filter(

# =========================================================================    ) \

# PASO 2: Aplicar row_number() sobre la ventana        row_number().over(window_spec)  # row_number() asigna 1, 2, 3, ... por ventana

# =========================================================================        "row_num",                  # Nueva columna: número de fila en la ventana

df_dedup = df_bronze \    .withColumn(

In [ ]:
# Estandarización
df_clean = df_dedup \
    .withColumn("product_name", upper(trim(col("product_name")))) \
    .withColumn("region", upper(trim(col("region")))) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("date", to_date(col("date")))

print("✅ Estandarización aplicada")
display(df_clean.limit(5))

In [ ]:
# Data Quality Checks
def validate_data_quality(df):
    """Valida calidad de datos y retorna df con columnas de validación"""
    df_validated = df \
        .withColumn("quality_issues", array()) \
        .withColumn("quality_issues",
                    when(col("transaction_id").isNull(), 
                         array_union(col("quality_issues"), array(lit("missing_transaction_id"))))
                    .otherwise(col("quality_issues"))) \
        .withColumn("quality_issues",
                    when(col("amount") <= 0, 
                         array_union(col("quality_issues"), array(lit("invalid_amount"))))
                    .otherwise(col("quality_issues"))) \
        .withColumn("quality_issues",
                    when(col("quantity") <= 0, 
                         array_union(col("quality_issues"), array(lit("invalid_quantity"))))
                    .otherwise(col("quality_issues"))) \
        .withColumn("quality_issues",
                    when(~col("status").isin(["completed", "pending", "cancelled"]), 
                         array_union(col("quality_issues"), array(lit("invalid_status"))))
                    .otherwise(col("quality_issues"))) \
        .withColumn("is_valid", size(col("quality_issues")) == 0)
    
    return df_validated

df_validated = validate_data_quality(df_clean)
valid_count = df_validated.filter(col("is_valid")).count()
invalid_count = df_validated.filter(~col("is_valid")).count()

print(f"✅ Registros válidos: {valid_count}")
print(f"⚠️  Registros inválidos: {invalid_count}")

In [ ]:
# Guardar en Silver (solo válidos)
table_silver = "lab04_silver_sales"
table_silver_errors = "lab04_silver_sales_errors"

df_silver = df_validated.filter(col("is_valid")).drop("quality_issues", "is_valid")
df_errors = df_validated.filter(~col("is_valid"))

df_silver.write.format("delta").mode("overwrite").saveAsTable(table_silver)
df_errors.write.format("delta").mode("overwrite").saveAsTable(table_silver_errors)

print(f"✅ Silver guardada: {df_silver.count()} registros válidos")
print(f"⚠️  Errores guardados: {df_errors.count()} registros")

## Parte 3: Capa Gold - Agregaciones de Negocio

In [ ]:
# Gold 1: Daily Sales by Region
df_daily_sales = df_silver \
    .groupBy("date", "region") \
    .agg(
        count("*").alias("num_transactions"),
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_transaction"),
        sum("quantity").alias("total_units"),
        countDistinct("customer_id").alias("unique_customers")
    ) \
    .orderBy("date", "region")

print(f"✅ Daily Sales: {df_daily_sales.count()} registros")
display(df_daily_sales.limit(10))

In [ ]:
# Gold 2: Top Products by Revenue
df_top_products = df_silver \
    .groupBy("product_code", "product_name") \
    .agg(
        sum("amount").alias("total_revenue"),
        sum("quantity").alias("total_quantity"),
        (sum("amount") / sum("quantity")).alias("avg_revenue_per_unit")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)

print("✅ Top 10 Products:")
display(df_top_products)

In [ ]:
# Gold 3: Customer RFM Segmentation
from pyspark.sql.functions import datediff, max as spark_max

df_rfm = df_silver \
    .filter(col("status") == "completed") \
    .groupBy("customer_id") \
    .agg(
        datediff(current_date(), spark_max("date")).alias("recency"),
        count("*").alias("frequency"),
        sum("amount").alias("monetary")
    ) \
    .withColumn("segment",
                when((col("recency") <= 7) & (col("frequency") >= 5) & (col("monetary") >= 2000), "VIP")
                .when((col("recency") <= 30) & (col("frequency") >= 3), "Active")
                .when((col("recency") <= 60) & (col("monetary") >= 1000), "Potential")
                .otherwise("At Risk"))

print("✅ Customer Segmentation:")
display(df_rfm.groupBy("segment").count().orderBy("count", ascending=False))

In [ ]:
# Guardar tablas Gold
gold_daily_path = "/tmp/lab04/gold/daily_sales_by_region"
gold_products_path = "/tmp/lab04/gold/top_products"
gold_rfm_path = "/tmp/lab04/gold/customer_rfm"

df_daily_sales.write.format("delta").mode("overwrite").save(gold_daily_path)
df_top_products.write.format("delta").mode("overwrite").save(gold_products_path)
df_rfm.write.format("delta").mode("overwrite").save(gold_rfm_path)

print("✅ Tablas Gold guardadas")

## Parte 4: Delta Lake Operations

In [ ]:
# MERGE - Upsert de datos
from delta.tables import DeltaTable

# Crear datos nuevos para merge
df_updates = generate_sales_data(50, "BATCH_003").select(
    "transaction_id", "date", "product_code", "product_name", 
    "region", "amount", "quantity", "customer_id", "status"
)

# Cargar Delta Table
delta_table = DeltaTable.forPath(spark, silver_path)

# Merge
delta_table.alias("target").merge(
    df_updates.alias("source"),
    "target.transaction_id = source.transaction_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print(f"✅ MERGE completado")

In [ ]:
# OPTIMIZE - Compactación de archivos
spark.sql(f"OPTIMIZE delta.`{silver_path}`")
print("✅ OPTIMIZE completado")

In [ ]:
# Z-ORDER - Indexación
spark.sql(f"OPTIMIZE delta.`{silver_path}` ZORDER BY (region, date, product_code)")
print("✅ Z-ORDER completado")

In [ ]:
# DESCRIBE HISTORY
history = spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`")
print("📜 Historial de versiones:")
display(history.select("version", "timestamp", "operation", "operationParameters").limit(10))

In [ ]:
# TIME TRAVEL - Leer versión anterior
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(silver_path)
df_latest = spark.read.format("delta").load(silver_path)

print(f"📊 Versión 0: {df_v0.count()} registros")
print(f"📊 Versión actual: {df_latest.count()} registros")
print(f"📈 Diferencia: {df_latest.count() - df_v0.count()} nuevos registros")

## ✅ Lab Completado

Has implementado:
- ✅ Arquitectura Medallion (Bronze → Silver → Gold)
- ✅ Deduplicación con window functions
- ✅ Data quality checks
- ✅ Agregaciones de negocio
- ✅ Delta Lake operations (MERGE, OPTIMIZE, Z-ORDER)
- ✅ Time Travel